Fitting a linear probe, uses average pooling instead of a CLS token

In [1]:
import torch
from torch import nn, optim
import torchvision
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
import numpy as np
from math import isqrt
import os
import json

In [2]:
from model import get_pos_embeddings, Net

In [3]:
MODEL_DIR = "checkpoints"
with open(os.path.join(MODEL_DIR, "config.json"), "r") as f:
    cfg = json.load(f)

In [4]:
SEED = 12
BATCH_SIZE = 128
CHECKPOINT_NAME = "checkpoint_epoch_400.pt"

In [5]:
# PRETRAINED MODEL PARAMETERS

# --- Base params (from config) ---
# Image
D_IMAGE    = cfg["data"]["d_image"]
N_CHANNELS = cfg["data"]["n_channels"]
PATCH_SIZE = cfg["data"]["patch_size"]
# Encoder
D_ENC            = cfg["model"]["d_enc"]
N_HEADS_ENC      = cfg["model"]["n_heads_enc"]
N_ENCODER_BLOCKS = cfg["model"]["n_encoder_blocks"]
MLP_RATIO        = cfg["model"]["mlp_ratio"]
# Decoder
D_DEC            = cfg["model"]["d_dec"]
N_HEADS_DEC      = cfg["model"]["n_heads_dec"]
N_DECODER_BLOCKS = cfg["model"]["n_decoder_blocks"]
# Masking (needed to reconstruct Net)
PERCENT_UNMASKED = cfg["metadata"]["percent_unmasked"]
# --- Derived params (computed) ---
D_PATCH   = (PATCH_SIZE ** 2) * N_CHANNELS
N_PATCHES = (D_IMAGE ** 2) // (PATCH_SIZE ** 2)
N_ROWS    = D_IMAGE // PATCH_SIZE
D_ENC_MLP = int(MLP_RATIO * D_ENC)
D_DEC_MLP = int(MLP_RATIO * D_DEC)

In [6]:
# Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(D_IMAGE, scale=(0.75, 1.0)), # Larger crops for linear probe
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4467, 0.4398, 0.4066), (0.2604, 0.2566, 0.2713))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4467, 0.4398, 0.4066), (0.2603, 0.2566, 0.2713))
])

In [7]:
# Download and load datasets
supervised_trainset = torchvision.datasets.STL10(
    root='./data', split='train', download=True, transform=train_transform
)
testset = torchvision.datasets.STL10(
    root='./data', split='test', download=True, transform=test_transform
)
# DataLoaders
testloader = DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

val_size = 500 # 10%
train_size = len(supervised_trainset) - val_size
generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(
    supervised_trainset, [train_size, val_size], generator=generator
)
supervised_trainloader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
supervised_valloader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

In [8]:
device = torch.device('cuda:0')

In [9]:
from model import get_pos_embeddings, Net
pos_embeddings_enc = get_pos_embeddings(D_ENC, N_PATCHES, N_ROWS)
pos_embeddings_dec = get_pos_embeddings(D_DEC, N_PATCHES, N_ROWS)

net = Net(
    n_encoder_blocks=N_ENCODER_BLOCKS,
    n_decoder_blocks=N_DECODER_BLOCKS,
    d_image=D_IMAGE,
    patch_size=PATCH_SIZE,
    d_patch=D_PATCH,
    n_patches=N_PATCHES,
    n_rows=N_ROWS,
    d_enc=D_ENC,
    d_enc_mlp=D_ENC_MLP,
    d_dec=D_DEC,
    d_dec_mlp=D_DEC_MLP,
    n_heads_enc=N_HEADS_ENC,
    n_heads_dec=N_HEADS_DEC,
    pos_embeddings_enc=pos_embeddings_enc,
    pos_embeddings_dec=pos_embeddings_dec,
    percent_unmasked=PERCENT_UNMASKED)

In [10]:
path = os.path.join(MODEL_DIR, CHECKPOINT_NAME)
sd = torch.load(path, map_location="cpu", weights_only=True)["model_state_dict"]
net.load_state_dict(sd)
net.to(device)

Net(
  (img2enc_projection): Linear(in_features=192, out_features=384, bias=True)
  (encoder_blocks): ModuleList(
    (0-11): 12 x ModuleDict(
      (norm_a): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (msa): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
      )
      (norm_b): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp_a): Linear(in_features=384, out_features=1536, bias=True)
      (mlp_b): Linear(in_features=1536, out_features=384, bias=True)
    )
  )
  (enc_terminal_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (enc2dec_projection): Linear(in_features=384, out_features=256, bias=True)
  (decoder_blocks): ModuleList(
    (0-7): 8 x ModuleDict(
      (norm_a): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (msa): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (n

# Linear Probe

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR, ConstantLR

net.eval()
N_CLASSES = 10
N_EPOCHS = 50

# LR ∈ [3e-4, 1e-3, 3e-3]
LR = 1e-3

# # Simple cosine decay
linear_probe = nn.Linear(D_ENC, N_CLASSES).to(device)
optimizer_probe = optim.AdamW(linear_probe.parameters(), lr=LR, weight_decay=0.0)
scheduler = CosineAnnealingLR(optimizer_probe, T_max=N_EPOCHS)

criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_state = None
best_epoch = -1

for epoch in range(N_EPOCHS):
    # --- train ---
    linear_probe.train()
    running_loss = 0.0
    n_batches = 0
    for i, data in enumerate(supervised_trainloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        with torch.no_grad():
            embeddings, _, _, _ = net.encode(inputs, mask=False)
            reps = embeddings.mean(dim=1)
        logits = linear_probe(reps)
        loss = criterion(logits, labels)

        optimizer_probe.zero_grad()
        loss.backward()
        optimizer_probe.step()

        running_loss += loss.item()
        n_batches += 1

        if i % 10 == 9:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.6f}')

    scheduler.step()
    train_loss = running_loss / n_batches

    # --- validate ---
    linear_probe.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in supervised_valloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            embeddings, _, _, _ = net.encode(inputs, mask=False)
            reps = embeddings.mean(dim=1)
            logits = linear_probe(reps)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    current_lr = optimizer_probe.param_groups[0]['lr']
    print(f'epoch {epoch + 1:3d} | train_loss={train_loss:.4f} | '
          f'val_acc={val_acc:.4f} | lr={current_lr:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().clone() for k, v in linear_probe.state_dict().items()}

# load best-validation weights back into the probe
linear_probe.load_state_dict(best_state)
linear_probe.eval()

print(f'Finished training. Best val_acc={best_val_acc:.4f} at epoch {best_epoch}')

In [ ]:
# Test the best linear probe
linear_probe.eval()
correct = 0
total = 0

with torch.no_grad():
    for i, data in enumerate(testloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        embeddings, _, _, _ = net.encode(inputs, mask=False)
        reps = embeddings.mean(dim=1)
        logits = linear_probe(reps)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f}")

# MLP

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR, ConstantLR

# Reload pretrained encoder so re-running this cell is deterministic.
path = os.path.join(MODEL_DIR, CHECKPOINT_NAME)
sd = torch.load(path, map_location="cpu", weights_only=True)["model_state_dict"]
net.load_state_dict(sd)
net.to(device)
net.eval()

N_CLASSES = 10
N_EPOCHS = 50

# encoder → global average pooling → LayerNorm
#         → Linear(384, 512)
#         → GELU
#         → Dropout
#         → Linear(512, 10)

# LR ∈ [1e-4, 3e-4, 1e-3]
LR = 1e-3
MLP_DIM = 512
WARMUP_EPOCHS = 10

mlp_probe = nn.Sequential(
    nn.LayerNorm(D_ENC),
    nn.Linear(D_ENC, MLP_DIM),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(MLP_DIM, N_CLASSES),
).to(device)

# CosineAnnealingLR with Warmup
optimizer_probe = optim.AdamW(mlp_probe.parameters(), lr=1e-3, weight_decay=1e-4)
warmup = LinearLR(optimizer_probe, start_factor=0.01, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(
    optimizer_probe, T_max=N_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6
)
scheduler = SequentialLR(
    optimizer_probe, [warmup, cosine], milestones=[WARMUP_EPOCHS]
)

criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_state = None
best_epoch = -1

for epoch in range(N_EPOCHS):
    # --- train ---
    mlp_probe.train()
    running_loss = 0.0
    n_batches = 0
    for i, data in enumerate(supervised_trainloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        with torch.no_grad():
            embeddings, _, _, _ = net.encode(inputs, mask=False)
            reps = embeddings.mean(dim=1)
        logits = mlp_probe(reps)
        loss = criterion(logits, labels)

        optimizer_probe.zero_grad()
        loss.backward()
        optimizer_probe.step()

        running_loss += loss.item()
        n_batches += 1

        if i % 10 == 9:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.6f}')

    scheduler.step()
    train_loss = running_loss / n_batches

    # --- validate ---
    mlp_probe.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in supervised_valloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            embeddings, _, _, _ = net.encode(inputs, mask=False)
            reps = embeddings.mean(dim=1)
            logits = mlp_probe(reps)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    current_lr = optimizer_probe.param_groups[0]['lr']
    print(f'epoch {epoch + 1:3d} | train_loss={train_loss:.4f} | '
          f'val_acc={val_acc:.4f} | lr={current_lr:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().clone() for k, v in mlp_probe.state_dict().items()}

# load best-validation weights back into the probe
mlp_probe.load_state_dict(best_state)
mlp_probe.eval()

print(f'Finished training. Best val_acc={best_val_acc:.4f} at epoch {best_epoch}')

In [ ]:
# Test the best MLP probe
mlp_probe.eval()
correct = 0
total = 0

with torch.no_grad():
    for i, data in enumerate(testloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        embeddings, _, _, _ = net.encode(inputs, mask=False)
        reps = embeddings.mean(dim=1)
        logits = mlp_probe(reps)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f}")

# Fine-Tune Last Transformer Block

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Reload pretrained encoder so re-running this cell is deterministic.
path = os.path.join(MODEL_DIR, CHECKPOINT_NAME)
sd = torch.load(path, map_location="cpu", weights_only=True)["model_state_dict"]
net.load_state_dict(sd)
net.to(device)

for p in net.parameters():
    p.requires_grad = False

last_block = net.encoder_blocks[-1]  # norm_a, msa, norm_b, mlp_a, mlp_b
for p in last_block.parameters():
    p.requires_grad = True
for p in net.enc_terminal_norm.parameters():
    p.requires_grad = True

trainable = [n for n, p in net.named_parameters() if p.requires_grad]
print(f"trainable encoder tensors ({len(trainable)}):")
for n in trainable:
    print(f"  {n}")

N_CLASSES = 10
N_EPOCHS = 50
WARMUP_EPOCHS = 5

# LR ∈ [3e-5, 1e-4, 3e-4, 1e-3]
LR = 1e-4 # 1e-4 gave 92% validation accuracy

cls_head = nn.Linear(D_ENC, N_CLASSES).to(device)

trainable_params = (
    list(last_block.parameters())
    + list(net.enc_terminal_norm.parameters())
    + list(cls_head.parameters())
)
optimizer = optim.AdamW(
    trainable_params,
    lr=LR,
    betas=(0.9, 0.999),
    weight_decay=0.05,
)
warmup = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer, T_max=N_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[WARMUP_EPOCHS])
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_state = None
best_epoch = -1

for epoch in range(N_EPOCHS):
    last_block.train()
    net.enc_terminal_norm.train()
    cls_head.train()
    running_loss = 0.0
    n_batches = 0
    for i, data in enumerate(supervised_trainloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        embeddings, _, _, _ = net.encode(inputs, mask=False)
        reps = embeddings.mean(dim=1)
        logits = cls_head(reps)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1
        if i % 10 == 9:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.6f}')

    scheduler.step()
    train_loss = running_loss / n_batches

    last_block.eval()
    net.enc_terminal_norm.eval()
    cls_head.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in supervised_valloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            embeddings, _, _, _ = net.encode(inputs, mask=False)
            reps = embeddings.mean(dim=1)
            logits = cls_head(reps)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    current_lr = optimizer.param_groups[0]["lr"]
    print(f'epoch {epoch + 1:3d} | train_loss={train_loss:.4f} | '
          f'val_acc={val_acc:.4f} | lr={current_lr:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_state = {
            "last_block": {k: v.detach().clone() for k, v in last_block.state_dict().items()},
            "enc_terminal_norm": {k: v.detach().clone() for k, v in net.enc_terminal_norm.state_dict().items()},
            "cls_head": {k: v.detach().clone() for k, v in cls_head.state_dict().items()},
        }

last_block.load_state_dict(best_state["last_block"])
net.enc_terminal_norm.load_state_dict(best_state["enc_terminal_norm"])
cls_head.load_state_dict(best_state["cls_head"])
last_block.eval()
net.enc_terminal_norm.eval()
cls_head.eval()

print(f'Finished training. Best val_acc={best_val_acc:.4f} at epoch {best_epoch}')

trainable encoder tensors (14):
  encoder_blocks.11.norm_a.weight
  encoder_blocks.11.norm_a.bias
  encoder_blocks.11.msa.in_proj_weight
  encoder_blocks.11.msa.in_proj_bias
  encoder_blocks.11.msa.out_proj.weight
  encoder_blocks.11.msa.out_proj.bias
  encoder_blocks.11.norm_b.weight
  encoder_blocks.11.norm_b.bias
  encoder_blocks.11.mlp_a.weight
  encoder_blocks.11.mlp_a.bias
  encoder_blocks.11.mlp_b.weight
  encoder_blocks.11.mlp_b.bias
  enc_terminal_norm.weight
  enc_terminal_norm.bias
[1,    10] loss: 2.334891
[1,    20] loss: 2.338394
[1,    30] loss: 2.334836
epoch   1 | train_loss=2.3324 | val_acc=0.0840 | lr=6.24e-05
[2,    10] loss: 2.307052
[2,    20] loss: 2.271889
[2,    30] loss: 2.241476
epoch   2 | train_loss=2.2263 | val_acc=0.3140 | lr=1.22e-04
[3,    10] loss: 2.062473
[3,    20] loss: 1.990591
[3,    30] loss: 1.920416
epoch   3 | train_loss=1.8762 | val_acc=0.6520 | lr=1.81e-04
[4,    10] loss: 1.425439
[4,    20] loss: 1.304723
[4,    30] loss: 1.193845
epoch  

In [14]:
# Test last-block + classification head
cls_head.eval()
correct = 0
total = 0

with torch.no_grad():
    for data in testloader:
        inputs, labels = data[0].to(device), data[1].to(device)
        embeddings, _, _, _ = net.encode(inputs, mask=False)
        reps = embeddings.mean(dim=1)
        logits = cls_head(reps)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.8670
